# Notebook 5: Model Compression
## EEEM073 – AI and Sustainability
### Project: Predicting Burned Area of Forest Fires Using Machine Learning

**Prerequisite:** Run Notebooks 1–4 first.

---
### Why Model Compression?
Smaller, faster models consume less energy during inference — directly supporting **sustainable AI**.
This notebook applies compression to the MLP and Transformer from Notebook 3.

**Three compression techniques (module Week 7 + reading list):**

| Technique | Reference | Applied To |
|---|---|---|
| **Dynamic Quantisation** | Krishnamoorthi (2018) | MLP + Transformer |
| **Weight Pruning** | Li et al. (2017) | MLP + Transformer |
| **Knowledge Distillation** | Hinton et al. (2015) | MLP → smaller MLP |

Each compressed model is compared against its baseline using MAE, RMSE, R², size, and inference time.


## 1. Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, time, copy, warnings, io, json
import joblib

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.utils.prune as prune
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
os.makedirs('outputs/compressed_models', exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# NOTE: We force CPU throughout this notebook.
# Quantisation only works on CPU in PyTorch.
# Pruning fine-tuning on CPU avoids device-mismatch errors with pruning masks.
device = torch.device('cpu')
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Load preprocessed data from Notebook 1
X_train = pd.read_csv('outputs/X_train.csv')
X_val   = pd.read_csv('outputs/X_val.csv')
X_test  = pd.read_csv('outputs/X_test.csv')
y_train = pd.read_csv('outputs/y_train.csv').squeeze()
y_val   = pd.read_csv('outputs/y_val.csv').squeeze()
y_test  = pd.read_csv('outputs/y_test.csv').squeeze()
feature_names = pd.read_csv('outputs/feature_names.csv')['feature'].tolist()
n_features = X_train.shape[1]

def to_tensor(df):
    """Convert DataFrame or array to PyTorch FloatTensor on CPU."""
    return torch.FloatTensor(df.values if hasattr(df, 'values') else df)

# Keep everything on CPU — required for quantisation compatibility
X_train_t = to_tensor(X_train)
X_val_t   = to_tensor(X_val)
X_test_t  = to_tensor(X_test)
y_train_t = to_tensor(y_train)
y_val_t   = to_tensor(y_val)

BATCH_SIZE   = 32
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t,   y_val_t),   batch_size=BATCH_SIZE, shuffle=False)

print(f"Data loaded — {n_features} features, {len(X_test)} test samples")

In [ ]:
# ── Rebuild model architectures (must be identical to Notebook 3) ────────────

class MLPRegressor(nn.Module):
    """
    Feedforward Neural Network — same architecture as Notebook 3.
    Input → [Linear → BatchNorm → ReLU → Dropout] × 3 → Output
    """
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout=0.3):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x).squeeze(-1)


class TabularTransformer(nn.Module):
    """
    Transformer-based tabular regressor — same architecture as Notebook 3.
    Reference: Nascimento et al. (2023), Energy, 278, 127678.

    FIX: The forward() disables the MHA fast-path during inference.
    This is required because dynamic quantisation replaces Linear layers with
    quantised equivalents that are not compatible with PyTorch's fused
    attention kernel. Without this fix, a RuntimeError is raised during
    backward() or inference on quantised/pruned Transformer models.
    """
    def __init__(self, input_dim, d_model=64, nhead=4,
                 num_encoder_layers=2, dim_feedforward=128, dropout=0.2):
        super().__init__()
        self.feature_embedding   = nn.Linear(1, d_model)
        self.positional_encoding = nn.Parameter(torch.randn(input_dim, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_encoder_layers
        )
        self.regression_head = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1)
        )

    def forward(self, x):
        x = x.unsqueeze(-1)
        x = self.feature_embedding(x) + self.positional_encoding.unsqueeze(0)
        # Disable fused attention fast-path — required after pruning or quantisation
        # because modified Linear weights are incompatible with the fused kernel.
        # This ensures correct backward() during fine-tuning of pruned models.
        for layer in self.transformer_encoder.layers:
            layer.self_attn.fast_path_enabled = False
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)
        return self.regression_head(x).squeeze(-1)


# Load trained baseline models — map to CPU explicitly
mlp_baseline = MLPRegressor(input_dim=n_features)
mlp_baseline.load_state_dict(torch.load('outputs/models/mlp.pt', map_location='cpu'))
mlp_baseline.eval()

tf_baseline = TabularTransformer(input_dim=n_features)
tf_baseline.load_state_dict(torch.load('outputs/models/transformer.pt', map_location='cpu'))
tf_baseline.eval()

print("Baseline models loaded on CPU.")
print(f"  MLP params:         {sum(p.numel() for p in mlp_baseline.parameters()):,}")
print(f"  Transformer params: {sum(p.numel() for p in tf_baseline.parameters()):,}")

## 2. Helper Functions

In [ ]:
def evaluate(y_true, y_pred, label=''):
    """
    Compute and print MAE, RMSE, R².
    Same metrics as Notebook 4 for consistent comparison.

    Args:
        y_true: Ground truth values
        y_pred: Model predictions
        label (str): Display name
    Returns:
        dict of metric values
    """
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"  {label:<42} MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}")
    return {'Model': label, 'MAE': mae, 'RMSE': rmse, 'R2': r2}


def get_preds(model, X_tensor):
    """
    Run inference and return numpy predictions.
    All tensors and models stay on CPU in this notebook.

    Args:
        model: PyTorch model (CPU)
        X_tensor: FloatTensor on CPU
    Returns:
        numpy array of predictions
    """
    model.eval()
    with torch.no_grad():
        return model(X_tensor.cpu()).cpu().numpy()


def measure_ms(model, X_tensor, n_runs=100):
    """
    Average inference time in milliseconds over n_runs.
    Repeated runs reduce timing noise from OS scheduling.

    Args:
        model: PyTorch model
        X_tensor: Input tensor
        n_runs (int): Number of timing repetitions
    Returns:
        float: Mean inference time in ms
    """
    model.eval()
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        with torch.no_grad():
            model(X_tensor.cpu())
        times.append(time.perf_counter() - t0)
    return np.mean(times) * 1000


def get_size_mb(model):
    """
    Serialise model state dict to a buffer and measure bytes.
    Gives actual on-disk size including sparse weights after pruning.
    """
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.tell() / 1e6


def sparsity_pct(model):
    """
    Percentage of Linear layer WEIGHTS that are exactly zero (pruned).
    Only counts weight tensors — not bias or other parameters.
    0% = dense (unpruned), 50% = half weights zeroed.
    """
    total, zeros = 0, 0
    for m in model.modules():
        if isinstance(m, nn.Linear):
            total += m.weight.numel()
            zeros += (m.weight == 0).sum().item()
    return round(zeros / total * 100, 1) if total > 0 else 0.0


# Storage for all results
all_results = []

# Record baseline metrics
print("=" * 68)
print("BASELINE PERFORMANCE (uncompressed — test set)")
print("=" * 68)

mlp_base_preds = get_preds(mlp_baseline, X_test_t)
tf_base_preds  = get_preds(tf_baseline,  X_test_t)

mlp_base_m = evaluate(y_test, mlp_base_preds, 'MLP Baseline')
tf_base_m  = evaluate(y_test, tf_base_preds,  'Transformer Baseline')

mlp_base_ms   = measure_ms(mlp_baseline, X_test_t)
tf_base_ms    = measure_ms(tf_baseline,  X_test_t)
mlp_base_size = get_size_mb(mlp_baseline)
tf_base_size  = get_size_mb(tf_baseline)

for m, ms, size in [(mlp_base_m, mlp_base_ms, mlp_base_size),
                    (tf_base_m,  tf_base_ms,  tf_base_size)]:
    all_results.append({**m, 'Technique': 'None (Baseline)',
                        'Inference_ms': round(ms, 3),
                        'Size_MB': round(size, 4),
                        'Sparsity_%': 0.0})

print(f"\n  MLP Baseline     — {mlp_base_ms:.2f} ms | {mlp_base_size:.4f} MB")
print(f"  Transformer Base — {tf_base_ms:.2f} ms | {tf_base_size:.4f} MB")

---
## 3. Technique 1 — Dynamic Quantisation

### What is Quantisation?
Quantisation converts weights from **32-bit float (FP32) → 8-bit integer (INT8)**:
- ~75% smaller model size
- Faster CPU inference (integer ops are cheaper than float ops)
- Minimal accuracy loss for well-trained models

Dynamic quantisation quantises weights at save time, activations at runtime — no retraining needed.

**Reference:** Krishnamoorthi, R. (2018). *Quantizing deep convolutional networks for efficient inference.* arXiv.

In [ ]:
print("=" * 68)
print("TECHNIQUE 1: Dynamic Quantisation (FP32 → INT8)")
print("=" * 68)

# Apply dynamic quantisation — targets all Linear layers.
# Models must be on CPU (already ensured above).
# We deep-copy first so the baseline remains unmodified.
mlp_quant = torch.quantization.quantize_dynamic(
    copy.deepcopy(mlp_baseline).cpu(), {nn.Linear}, dtype=torch.qint8
)
tf_quant = torch.quantization.quantize_dynamic(
    copy.deepcopy(tf_baseline).cpu(),  {nn.Linear}, dtype=torch.qint8
)
mlp_quant.eval()
tf_quant.eval()

print("Quantisation applied. Evaluating...")
mlp_q_preds = get_preds(mlp_quant, X_test_t)
tf_q_preds  = get_preds(tf_quant,  X_test_t)

print("\nQuantised Model Performance:")
mlp_q_m = evaluate(y_test, mlp_q_preds, 'MLP — Quantised (INT8)')
tf_q_m  = evaluate(y_test, tf_q_preds,  'Transformer — Quantised (INT8)')

mlp_q_ms   = measure_ms(mlp_quant, X_test_t)
tf_q_ms    = measure_ms(tf_quant,  X_test_t)
mlp_q_size = get_size_mb(mlp_quant)
tf_q_size  = get_size_mb(tf_quant)

print(f"\nMLP:")
print(f"  Size:  {mlp_base_size:.4f} MB  →  {mlp_q_size:.4f} MB  ({(1-mlp_q_size/mlp_base_size)*100:.1f}% smaller)")
print(f"  Speed: {mlp_base_ms:.2f} ms  →  {mlp_q_ms:.2f} ms")
print(f"  RMSE change: {mlp_q_m['RMSE'] - mlp_base_m['RMSE']:+.4f}")
print(f"Transformer:")
print(f"  Size:  {tf_base_size:.4f} MB  →  {tf_q_size:.4f} MB  ({(1-tf_q_size/tf_base_size)*100:.1f}% smaller)")
print(f"  Speed: {tf_base_ms:.2f} ms  →  {tf_q_ms:.2f} ms")
print(f"  RMSE change: {tf_q_m['RMSE'] - tf_base_m['RMSE']:+.4f}")

torch.save(mlp_quant.state_dict(), 'outputs/compressed_models/mlp_quantised.pt')
torch.save(tf_quant.state_dict(),  'outputs/compressed_models/transformer_quantised.pt')

for m, ms, size in [(mlp_q_m, mlp_q_ms, mlp_q_size),
                    (tf_q_m,  tf_q_ms,  tf_q_size)]:
    all_results.append({**m, 'Technique': 'Quantisation (INT8)',
                        'Inference_ms': round(ms, 3),
                        'Size_MB': round(size, 4),
                        'Sparsity_%': 0.0})

---
## 4. Technique 2 — Unstructured Weight Pruning

### What is Pruning?
Pruning sets the smallest-magnitude weights to zero — creating a **sparse model**.

We apply **L1 unstructured pruning at 50% sparsity** — half of all Linear layer weights are zeroed. After pruning, we fine-tune briefly to recover accuracy, then **make pruning permanent** to remove mask buffers.

**Key fix vs original:** We only prune **weights** (not biases). Pruning biases in PyTorch creates a second mask buffer that conflicts with BatchNorm and the Transformer's internal attention layers during `backward()`, causing `RuntimeError`. Weights carry the majority of model capacity so pruning weights-only is standard practice.

**Reference:** Li, H., et al. (2017). *Pruning Filters for Efficient ConvNets.* arXiv.

In [ ]:
def apply_pruning(model, amount=0.5):
    """
    Apply L1 unstructured global pruning to all Linear layer WEIGHTS only.

    We use global_unstructured (not per-layer) so the pruning threshold is
    determined across ALL layers together. This avoids over-pruning small
    layers and under-pruning large ones.

    IMPORTANT: We prune weights only (not bias). Pruning bias tensors in
    PyTorch creates mask buffers that break autograd during fine-tuning of
    models with BatchNorm or Transformer attention layers.

    Args:
        model: PyTorch model (will NOT be modified in-place — caller passes a deepcopy)
        amount (float): Fraction of weights to zero out (e.g. 0.5 = 50%)
    Returns:
        Model with pruning masks applied (not yet permanent)
    """
    # Collect (module, parameter_name) pairs for weights only
    params_to_prune = [
        (module, 'weight')
        for module in model.modules()
        if isinstance(module, nn.Linear)
    ]
    # Global L1 unstructured pruning — single threshold across all layers
    prune.global_unstructured(
        params_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=amount
    )
    return model


def make_permanent(model):
    """
    Remove pruning mask buffers and bake zero weights permanently.
    During pruning, PyTorch stores original_weight + binary mask.
    Making permanent removes the mask, reducing memory footprint.

    Args:
        model: Model with active pruning masks
    Returns:
        Model with masks removed and sparse weights baked in
    """
    for module in model.modules():
        if isinstance(module, nn.Linear):
            try:
                prune.remove(module, 'weight')
            except ValueError:
                pass  # already permanent or not pruned
    return model


def finetune(model, train_loader, val_loader, n_epochs=30, lr=5e-4):
    """
    Fine-tune a pruned model to recover accuracy lost during pruning.
    The remaining non-zero weights adjust to compensate for zeroed ones.

    FIX: model.to(device) is called inside this function to ensure
    all tensors (including pruning mask buffers) are on the same device
    before any forward/backward pass. Mismatched devices between the
    mask buffer and input tensor caused the original RuntimeError.

    Uses early stopping on validation loss to prevent overfitting.

    Args:
        model: Pruned PyTorch model
        train_loader, val_loader: PyTorch DataLoaders
        n_epochs (int): Maximum fine-tuning epochs
        lr (float): Learning rate (lower than original training)
    Returns:
        Fine-tuned model with best validation weights restored
    """
    # Move model and all its buffers (including prune masks) to device
    model = model.to(device)
    model.train()

    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )
    criterion = nn.MSELoss()
    best_val, best_state, patience, p_count = float('inf'), None, 10, 0

    for epoch in range(n_epochs):
        model.train()
        for X_b, y_b in train_loader:
            # Both tensors already on CPU (device = cpu)
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = sum(
                criterion(model(X_b.to(device)), y_b.to(device)).item()
                for X_b, y_b in val_loader
            ) / len(val_loader)

        if val_loss < best_val:
            best_val   = val_loss
            # Deep copy state including mask buffers
            best_state = copy.deepcopy(model.state_dict())
            p_count    = 0
        else:
            p_count += 1
        if p_count >= patience:
            break

    if best_state:
        model.load_state_dict(best_state)
    return model


print("Pruning utilities defined. (Bug fixes: weights-only pruning + device-safe finetune)")

In [ ]:
print("=" * 68)
print("TECHNIQUE 2: L1 Unstructured Pruning (50% sparsity)")
print("=" * 68)

# ── Prune and fine-tune MLP ────────────────────────────────────────────────
print("\nPruning MLP (50%)...")
mlp_pruned = copy.deepcopy(mlp_baseline)
mlp_pruned = apply_pruning(mlp_pruned, amount=0.5)
print(f"  Pre-finetune sparsity: {sparsity_pct(mlp_pruned)}%")
print("  Fine-tuning pruned MLP (30 epochs)...")
mlp_pruned = finetune(mlp_pruned, train_loader, val_loader, n_epochs=30)
mlp_pruned = make_permanent(mlp_pruned)

mlp_p_preds = get_preds(mlp_pruned, X_test_t)
mlp_p_m     = evaluate(y_test, mlp_p_preds, 'MLP — Pruned (50%)')
mlp_p_ms    = measure_ms(mlp_pruned, X_test_t)
mlp_p_size  = get_size_mb(mlp_pruned)
mlp_p_spar  = sparsity_pct(mlp_pruned)

print(f"  Final sparsity: {mlp_p_spar}%")
print(f"  Size:  {mlp_base_size:.4f} MB  →  {mlp_p_size:.4f} MB  ({(1-mlp_p_size/mlp_base_size)*100:.1f}% smaller)")
print(f"  Speed: {mlp_base_ms:.2f} ms  →  {mlp_p_ms:.2f} ms")
print(f"  RMSE change: {mlp_p_m['RMSE'] - mlp_base_m['RMSE']:+.4f}")

torch.save(mlp_pruned.state_dict(), 'outputs/compressed_models/mlp_pruned.pt')
all_results.append({**mlp_p_m, 'Technique': 'Pruning (L1, 50%)',
                    'Inference_ms': round(mlp_p_ms, 3),
                    'Size_MB': round(mlp_p_size, 4),
                    'Sparsity_%': mlp_p_spar})

In [ ]:
# ── Prune and fine-tune Transformer ────────────────────────────────────────
print("Pruning Transformer (50%)...")
tf_pruned = copy.deepcopy(tf_baseline)
tf_pruned = apply_pruning(tf_pruned, amount=0.5)
print(f"  Pre-finetune sparsity: {sparsity_pct(tf_pruned)}%")
print("  Fine-tuning pruned Transformer (30 epochs)...")
tf_pruned = finetune(tf_pruned, train_loader, val_loader, n_epochs=30)
tf_pruned = make_permanent(tf_pruned)

tf_p_preds = get_preds(tf_pruned, X_test_t)
tf_p_m     = evaluate(y_test, tf_p_preds, 'Transformer — Pruned (50%)')
tf_p_ms    = measure_ms(tf_pruned, X_test_t)
tf_p_size  = get_size_mb(tf_pruned)
tf_p_spar  = sparsity_pct(tf_pruned)

print(f"  Final sparsity: {tf_p_spar}%")
print(f"  Size:  {tf_base_size:.4f} MB  →  {tf_p_size:.4f} MB  ({(1-tf_p_size/tf_base_size)*100:.1f}% smaller)")
print(f"  Speed: {tf_base_ms:.2f} ms  →  {tf_p_ms:.2f} ms")
print(f"  RMSE change: {tf_p_m['RMSE'] - tf_base_m['RMSE']:+.4f}")

torch.save(tf_pruned.state_dict(), 'outputs/compressed_models/transformer_pruned.pt')
all_results.append({**tf_p_m, 'Technique': 'Pruning (L1, 50%)',
                    'Inference_ms': round(tf_p_ms, 3),
                    'Size_MB': round(tf_p_size, 4),
                    'Sparsity_%': tf_p_spar})

---
## 5. Technique 3 — Knowledge Distillation

### What is Knowledge Distillation?
A **teacher model** (large MLP) trains a **student model** (smaller MLP) by transferring its "knowledge" through soft predictions. The student learns both from the true labels and from the teacher's outputs simultaneously.

**Loss = α × distillation_loss + (1−α) × task_loss**
- α = 0.5 balances learning from the teacher vs learning from the labels

The student is ~3× smaller and faster, but retains most of the teacher's accuracy.

**Reference:** Hinton, G., Vinyals, O., & Dean, J. (2015). *Distilling the Knowledge in a Neural Network.* arXiv.

In [ ]:
print("=" * 68)
print("TECHNIQUE 3: Knowledge Distillation (Teacher MLP → Student MLP)")
print("=" * 68)

class StudentMLP(nn.Module):
    """
    Lightweight student MLP — approximately 3× fewer parameters than teacher.
    Teacher hidden dims: [128, 64, 32]  →  Student hidden dims: [64, 32, 16]
    This reduction demonstrates that a much smaller model can retain
    most of the teacher's predictive capability via distillation.
    """
    def __init__(self, input_dim, hidden_dims=[64, 32, 16], dropout=0.2):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x).squeeze(-1)


def distil_loss(s_out, t_out, y_true, alpha=0.5):
    """
    Combined knowledge distillation loss for regression.

    task_loss:   student vs ground truth  (ensures accuracy on real labels)
    distil_loss: student vs teacher soft outputs (transfers teacher knowledge)
    alpha controls the balance between both objectives.

    Reference: Hinton et al. (2015)

    Args:
        s_out: Student predictions tensor
        t_out: Teacher predictions tensor (detached — no gradient)
        y_true: Ground truth labels tensor
        alpha (float): Weight for distillation loss (0=task only, 1=distil only)
    Returns:
        Combined scalar loss tensor
    """
    mse = nn.MSELoss()
    return alpha * mse(s_out, t_out.detach()) + (1 - alpha) * mse(s_out, y_true)


# Teacher = full MLP from Notebook 3 (frozen — not trained further)
teacher = mlp_baseline.to(device)
teacher.eval()

# Student = smaller MLP initialised randomly
student = StudentMLP(input_dim=n_features).to(device)

t_params = sum(p.numel() for p in teacher.parameters())
s_params = sum(p.numel() for p in student.parameters())
print(f"Teacher params: {t_params:,}")
print(f"Student params: {s_params:,}  ({s_params/t_params*100:.1f}% of teacher size)")
print("\nStudent and distillation loss defined.")

In [ ]:
# Train student via distillation
optimizer  = optim.Adam(student.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

n_epochs    = 150
patience    = 25
best_val    = float('inf')
best_state  = None
p_count     = 0
d_train_log = []
d_val_log   = []

print("Training student via knowledge distillation...")
start = time.time()

for epoch in range(n_epochs):
    student.train()
    train_loss = 0.0

    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)

        # Teacher produces soft targets — gradients NOT needed for teacher
        with torch.no_grad():
            t_out = teacher(X_b)

        s_out = student(X_b)
        loss  = distil_loss(s_out, t_out, y_b, alpha=0.5)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    student.eval()
    with torch.no_grad():
        val_loss = sum(
            nn.MSELoss()(student(X_b.to(device)), y_b.to(device)).item()
            for X_b, y_b in val_loader
        ) / len(val_loader)

    d_train_log.append(train_loss)
    d_val_log.append(val_loss)
    scheduler.step(val_loss)

    if val_loss < best_val:
        best_val   = val_loss
        best_state = copy.deepcopy(student.state_dict())
        p_count    = 0
    else:
        p_count += 1
    if p_count >= patience:
        print(f"  Early stopping at epoch {epoch+1}")
        break
    if (epoch + 1) % 30 == 0:
        print(f"  Epoch {epoch+1:>3}: train={train_loss:.4f}  val={val_loss:.4f}")

student.load_state_dict(best_state)
distil_time = time.time() - start
print(f"  Distillation complete in {distil_time:.1f}s")

In [ ]:
# Evaluate student
student.eval()
s_preds  = get_preds(student, X_test_t)
s_m      = evaluate(y_test, s_preds, 'Student MLP — Distilled')
s_ms     = measure_ms(student, X_test_t)
s_size   = get_size_mb(student)

print(f"\nDistillation Results:")
print(f"  Teacher size: {mlp_base_size:.4f} MB  →  Student: {s_size:.4f} MB  ({(1-s_size/mlp_base_size)*100:.1f}% smaller)")
print(f"  Teacher speed: {mlp_base_ms:.2f} ms  →  Student: {s_ms:.2f} ms")
print(f"  Teacher RMSE: {mlp_base_m['RMSE']:.4f}  →  Student RMSE: {s_m['RMSE']:.4f}")

torch.save(student.state_dict(), 'outputs/compressed_models/student_distilled.pt')
all_results.append({**s_m, 'Technique': 'Knowledge Distillation',
                    'Inference_ms': round(s_ms, 3),
                    'Size_MB': round(s_size, 4),
                    'Sparsity_%': 0.0})

## 6. Full Compression Comparison

In [ ]:
comp_df = pd.DataFrame(all_results)

# Rename columns for display
comp_df = comp_df.rename(columns={
    'Inference_ms': 'Inference (ms)',
    'Size_MB':      'Size (MB)',
    'Sparsity_%':   'Sparsity (%)'
})

print("=" * 105)
print("FULL COMPRESSION COMPARISON TABLE")
print("=" * 105)
print(comp_df[['Model','Technique','MAE','RMSE','R2','Inference (ms)','Size (MB)','Sparsity (%)']].to_string(index=False))

comp_df.to_csv('outputs/compression_comparison.csv', index=False)
print("\nSaved to outputs/compression_comparison.csv")

In [ ]:
# Visualisation 1 — RMSE, Size, Inference time bar charts
labels  = comp_df['Model'].tolist()
palette = ['#1a3a5c','#7b2d00','#5b9bd5','#d4703b','#90c4e4','#f4b382','#2ca02c']
colors  = (palette * 3)[:len(labels)]  # repeat palette if needed

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
metrics_to_plot = [
    ('RMSE',           'Figure 21a: Test RMSE (lower = better)',            'RMSE'),
    ('Size (MB)',      'Figure 21b: Model Size (MB) (lower = sustainable)',  'Size (MB)'),
    ('Inference (ms)', 'Figure 21c: Inference Time (ms) (lower = faster)',   'Milliseconds')
]

for ax, (col, title, ylabel) in zip(axes, metrics_to_plot):
    vals = comp_df[col].tolist()
    bars = ax.bar(range(len(labels)), vals, color=colors, edgecolor='white', alpha=0.9)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=38, ha='right', fontsize=8)
    ax.set_title(title, fontweight='bold', fontsize=9)
    ax.set_ylabel(ylabel)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.01,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7)

plt.suptitle('Figure 21: Model Compression — Performance, Size, and Speed Comparison',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/fig21_compression_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualisation 2 — Accuracy vs Size trade-off scatter
fig, ax = plt.subplots(figsize=(10, 6))

for i, row in comp_df.iterrows():
    c = colors[i] if i < len(colors) else '#999999'
    ax.scatter(row['Size (MB)'], row['RMSE'],
               color=c, s=200, zorder=5,
               edgecolors='black', linewidths=0.8)
    ax.annotate(
        row['Model'].replace(' — ', '\n'),
        (row['Size (MB)'], row['RMSE']),
        textcoords='offset points', xytext=(7, 4), fontsize=7.5
    )

ax.set_xlabel('Model Size (MB) — smaller is more sustainable', fontsize=11)
ax.set_ylabel('Test RMSE — lower is better', fontsize=11)
ax.set_title(
    'Figure 22: Accuracy vs Model Size Trade-off\n'
    'Ideal models sit in the bottom-left (small AND accurate)',
    fontsize=11, fontweight='bold'
)
ax.axvline(comp_df['Size (MB)'].median(), color='green',
           linestyle=':', alpha=0.5, label='Median size')
ax.axhline(comp_df['RMSE'].median(), color='red',
           linestyle=':', alpha=0.5, label='Median RMSE')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('outputs/fig22_accuracy_vs_size.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualisation 3 — Knowledge distillation training curve
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(d_train_log, label='Student Train Loss', color='#2ca02c', linewidth=1.5)
ax.plot(d_val_log,   label='Student Val Loss',   color='#d62728', linewidth=1.5, linestyle='--')
ax.set_title('Figure 23: Knowledge Distillation — Student Training and Validation Loss',
             fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_ylim(bottom=0)
ax.legend()
plt.tight_layout()
plt.savefig('outputs/fig23_distillation_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Sustainability Summary

In [ ]:
print("=" * 70)
print("SUSTAINABILITY DISCUSSION — COMPRESSION FINDINGS")
print("=" * 70)

base_lookup = {
    'MLP':         (mlp_base_m['RMSE'], mlp_base_ms, mlp_base_size),
    'Transformer': (tf_base_m['RMSE'],  tf_base_ms,  tf_base_size),
    'Student':     (mlp_base_m['RMSE'], mlp_base_ms, mlp_base_size)
}

for _, row in comp_df.iterrows():
    if 'Baseline' in row['Model']:
        continue
    key = 'Transformer' if 'Transformer' in row['Model'] else \
          'Student'     if 'Student'     in row['Model'] else 'MLP'
    base_rmse, base_ms, base_size = base_lookup[key]

    size_red  = (1 - row['Size (MB)']       / base_size)  * 100
    speed_imp = (1 - row['Inference (ms)']  / base_ms)    * 100
    rmse_chg  = (row['RMSE'] - base_rmse)   / base_rmse   * 100

    print(f"\n  [{row['Technique']}] {row['Model']}")
    print(f"    Size reduction:  {size_red:+.1f}%")
    print(f"    Speed change:    {speed_imp:+.1f}%")
    print(f"    RMSE change:     {rmse_chg:+.1f}%  ({'minimal' if abs(rmse_chg)<5 else 'noticeable'} accuracy trade-off)")

print("""
KEY INSIGHT FOR REPORT:
Compression reduces model size and inference time at a small accuracy cost.
For deployment in resource-constrained settings (IoT sensors, edge devices,
early wildfire warning systems), compressed models offer a better
performance-per-watt ratio — directly supporting Sustainable AI (SDG 9)
and enabling real-time risk prediction in remote forest environments.
""")

print("Figures saved to outputs/:")
for f in sorted(os.listdir('outputs')):
    if f.startswith('fig2'):
        print(f"  {f}")

print("\nCompressed models saved to outputs/compressed_models/:")
for f in sorted(os.listdir('outputs/compressed_models')):
    sz = os.path.getsize(f'outputs/compressed_models/{f}') / 1e3
    print(f"  {f}  ({sz:.1f} KB)")

print("\nAll 5 notebooks complete. Proceed to writing your report.")